# Clase 13 — Pipelines de ML con Prefect

**Módulo:** Ciclo de vida y MLOps

En las clases anteriores construimos un proceso que descarga o carga datos, prepara features, ajusta hiperparámetros, entrena modelos, calcula métricas y registra versiones. Hasta ahora ese proceso ha vivido principalmente en notebooks y scripts que dependen de que una persona respete el orden correcto.

En esta clase convertiremos ese recorrido en un **workflow explícito**. Prefect coordinará las etapas; `sklearn.pipeline.Pipeline` seguirá siendo responsable de aplicar el mismo preprocesamiento durante entrenamiento e inferencia; MLflow conservará experimentos, modelos y aliases.

**Resultado observable:** ejecutar un flow que reutiliza o descarga marzo y abril, prepara ambos meses con el mismo contrato, ejecuta o reutiliza HPO, entrena tres familias, reserva abril para evaluación final y deja visible el estado de cada etapa.

## Antes de comenzar

Desde la raíz del repositorio público revisa que no tengas trabajo sin guardar y actualiza el material:

```bash
git status
git pull
uv sync --locked
```

`uv sync --locked` instala exactamente las versiones declaradas por el profesor, incluida Prefect. No ejecutes `uv add` ni modifiques `pyproject.toml` o `uv.lock` en el repositorio público.

La práctica se realiza en `labs/trabajo-local/clase-13/`. Esa carpeta es local, está ignorada por Git y no constituye una entrega.

# 0. Antes de orquestar: programación y observabilidad

## 0.1. Cron: programar la ejecución de un comando

**Cron** es un programador de trabajos (*job scheduler*) tradicional de sistemas tipo Unix. Permite solicitar que un comando o script se ejecute periódicamente en minutos, horas, días o meses determinados. El nombre suele asociarse con *Chronos*, la palabra griega para tiempo.

Cada regla se conoce como **cron job** y normalmente se guarda en una tabla llamada **crontab**. Una expresión tiene cinco campos temporales seguidos del comando:

```text
┌──────── minuto (0–59)
│ ┌────── hora (0–23)
│ │ ┌──── día del mes (1–31)
│ │ │ ┌── mes (1–12)
│ │ │ │ ┌ día de la semana (0–6; normalmente 0 es domingo)
│ │ │ │ │
* * * * * comando
```

El asterisco significa «cualquier valor» y la coma enumera valores separados. Los dos operadores que suelen confundirse son `-` y `/`:

- El **guion `-`** selecciona un rango continuo e inclusivo. `1-5` en el día de la semana significa de lunes a viernes; `8-18` en la hora incluye todas las horas desde las 08 hasta las 18.
- La **diagonal `/`** aplica un paso: toma cada *n* valores dentro de `*` o de un rango. `*/15` en minutos significa cada 15 minutos; `8-18/2` en horas significa cada 2 horas dentro del rango 8–18. No es una división y `*/15` no significa solamente «en el minuto 15».

Ejemplos completos:

```text
0 6 * * *       todos los días a las 06:00
30 8 * * 1-5    a las 08:30, de lunes a viernes
*/20 * * * *    cada 20 minutos
0 8-18/2 * * 1-5  cada 2 horas, de 08:00 a 18:00, de lunes a viernes
```

### Ejercicio — Construir cinco expresiones de cron

Abre [crontab.guru](https://crontab.guru/). Para cada requisito escribe una expresión de cinco campos y comprueba que la explicación mostrada por la página coincida con lo solicitado. No agregues un comando y no configures una crontab real.

1. **Nivel 1 — valores fijos:** todos los días a las 06:35.
2. **Nivel 2 — lista:** lunes, miércoles y viernes a las 08:10.
3. **Nivel 3 — rango e intervalo:** cada 20 minutos, desde las 08:00 hasta las 18:40, todos los días.
4. **Nivel 4 — dos rangos:** cada 2 horas desde las 08:00 hasta las 18:00, únicamente de lunes a viernes.
5. **Nivel 5 — combinación:** en los minutos 12 y 42 de cada tercera hora entre las 06 y las 21, los días 1 y 15 de febrero, mayo, agosto y noviembre.

Para cada nivel anota la expresión de cinco campos y, en una oración propia, explica cuándo se ejecutaría. La dificultad aumenta porque cada nivel incorpora un operador o campo adicional.

Para cerrar, explica por qué incluso una expresión temporal correcta no garantiza que un modelo sólo se registre después de terminar correctamente su evaluación.

## 0.2. Logging: registrar qué ocurrió y con qué gravedad

A medida que un script crece necesitamos observarlo incluso cuando nadie está mirando la terminal. `print()` escribe texto, pero no incorpora de forma consistente severidad, hora, origen o destino. El módulo `logging` de la biblioteca estándar representa cada evento como un registro estructurado.

### Niveles de severidad

| Nivel | Valor | Uso típico | Ejemplo en nuestro pipeline |
|---|---:|---|---|
| `DEBUG` | 10 | detalle para diagnóstico | parámetros y dimensiones intermedias |
| `INFO` | 20 | progreso esperado | «abril ya existe; se reutiliza» |
| `WARNING` | 30 | situación inesperada pero recuperable | quedaron pocas filas después del filtro |
| `ERROR` | 40 | una operación no pudo completarse | el Parquet no puede abrirse |
| `CRITICAL` | 50 | el proceso completo está comprometido | no existe configuración válida |

El nivel configurado funciona como umbral. Con `level=logging.WARNING` se conservan `WARNING`, `ERROR` y `CRITICAL`, pero se omiten `DEBUG` e `INFO`. Esto permite aumentar el detalle durante una depuración sin cambiar cada llamada.

### Logger, handler y formatter

- Un **logger** recibe los eventos. Conviene nombrarlo por módulo o componente, por ejemplo `nyc-taxi.pipeline`.
- Un **handler** decide el destino: consola, archivo u otro sistema.
- Un **formatter** decide cómo se representa cada evento.

Campos comunes del formato:

| Campo | Significado |
|---|---|
| `%(asctime)s` | fecha y hora del evento |
| `%(levelname)s` | nivel como texto |
| `%(name)s` | nombre del logger |
| `%(message)s` | mensaje producido por el código |
| `%(funcName)s` | función que emitió el evento |

`datefmt` usa códigos de `strftime`: `%Y` año, `%m` mes, `%d` día, `%H` hora, `%M` minuto y `%S` segundo. Un formato como `%Y-%m-%d %H:%M:%S` evita fechas ambiguas.

### Excepciones y datos sensibles

Dentro de un bloque `except`, `logger.exception()` registra el mensaje y el traceback. Registrar la excepción no significa ignorarla: si la etapa no puede recuperarse, debe volver a lanzar el error para que Prefect marque la task como `Failed`. Nunca registres tokens, `.env`, correos, URLs privadas, rutas personales completas ni filas sensibles.

In [ ]:
import logging

# Un nombre permite reconocer qué componente produjo cada mensaje.
logger = logging.getLogger("nyc-taxi.pipeline")
# El logger acepta eventos desde DEBUG; el handler decidirá qué muestra.
logger.setLevel(logging.DEBUG)
# En un notebook la celda puede ejecutarse varias veces. Limpiar evita duplicados.
logger.handlers.clear()
# Evita que el mismo evento también suba al logger raíz del notebook.
logger.propagate = False

# StreamHandler envía los registros a la consola.
handler = logging.StreamHandler()
# Este destino muestra INFO y niveles más graves, pero oculta DEBUG.
handler.setLevel(logging.INFO)
# El formatter fija los campos y un formato de fecha no ambiguo.
handler.setFormatter(
    logging.Formatter(
        "%(asctime)s - %(levelname)s - %(name)s - %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
)
# Conectamos el destino configurado con nuestro logger.
logger.addHandler(handler)

# Compara el resultado: DEBUG no aparece; INFO, WARNING y ERROR sí.
logger.debug("Este detalle no aparece porque el handler inicia en INFO")
logger.info("Inicio de la preparación de marzo")
logger.warning("Quedaron pocas filas después del filtro")
logger.error("Ejemplo de una operación que no pudo completarse")

# 1. Machine Learning Pipeline o workflow de entrenamiento

Un **workflow de machine learning** es una secuencia coordinada de etapas que transforma datos en un modelo evaluado y utilizable. Cada etapa tiene entradas, trabajo, salidas y un criterio de éxito.

Nuestro recorrido actual ya contiene un workflow, aunque todavía esté repartido entre celdas y scripts:

1. obtener marzo y abril;
2. preparar el target y las cinco features;
3. dividir marzo para tuning;
4. buscar hiperparámetros para tres familias;
5. reentrenar cada familia con marzo completo;
6. evaluar una vez con abril;
7. registrar los modelos finales;
8. ordenar resultados y asignar aliases.

En un notebook, el orden visible de las celdas no garantiza el orden real en que fueron ejecutadas. Una variable puede conservar un valor viejo, una celda puede omitirse o abril puede consultarse antes de tiempo. Esto produce el problema del **notebook gigante**: funciona en la sesión de su autor, pero resulta difícil de repetir, mantener y diagnosticar.

In [ ]:
def obtener_datos():
    ...


def preparar_datos(rutas):
    ...


def ajustar_modelos(marzo):
    ...


def entrenar_y_evaluar(parametros, marzo, abril):
    ...


def registrar_modelos(resultados):
    ...


def main():
    rutas = obtener_datos()
    marzo, abril = preparar_datos(rutas)
    parametros = ajustar_modelos(marzo)
    resultados = entrenar_y_evaluar(parametros, marzo, abril)
    return registrar_modelos(resultados)

Convertir celdas en funciones mejora mucho el código: hace visibles las responsabilidades y permite probar piezas aisladas. Sin embargo, todavía quedan preguntas operativas:

- ¿qué función se está ejecutando y cuánto tarda?
- ¿qué ocurrió en la última ejecución?
- ¿cómo distinguimos una falla de red de un schema inválido?
- ¿qué etapa puede reintentarse sin repetir todo?
- ¿cómo representamos dependencias y parámetros de una ejecución?

Existen varios orquestadores: Apache Airflow, Dagster, Kubeflow Pipelines, Mage y Prefect, entre otros. En esta clase usaremos **Prefect** porque permite comenzar con funciones normales de Python y agregar orquestación mediante decoradores. No realizaremos una comparación exhaustiva de plataformas ni adoptaremos infraestructura de producción.

## 1.1. Dos significados de pipeline

| Objeto | Alcance | Ejemplo |
|---|---|---|
| `sklearn.pipeline.Pipeline` | transformaciones aprendidas y estimador | one-hot de zonas + Random Forest |
| Workflow de ML | ciclo completo de datos a registro | descarga → HPO → evaluación → Registry |

El `Pipeline` de scikit-learn evita que entrenamiento e inferencia apliquen transformaciones diferentes. El workflow evita que el proceso completo dependa de pasos manuales u orden oculto. Prefect no reemplaza a scikit-learn y scikit-learn no reemplaza a Prefect.

# 2. Prefect

## 2.1. Task

Una **task** es una unidad de trabajo observable dentro de un flow. Se crea agregando `@task` a una función de Python. Puede recibir entradas, producir una salida y terminar en un estado como `Completed` o `Failed`.

Una frontera de task es útil cuando necesitamos observar duración y estado, aplicar reintentos o reconocer una falla de manera independiente. No toda función debe ser task: `construir_pipeline()` es una función pura pequeña y puede seguir siendo una función normal.

```python
from prefect import task

@task
def contar_filas(datos):
    return len(datos)
```

Las tasks pueden registrar logs, usar reintentos, timeouts y caché. Esas capacidades no deben activarse por costumbre. Una descarga puede fallar transitoriamente y sí justifica un reintento; una columna faltante no se corrige repitiendo la misma función.

## 2.2. Flow

Un **flow** describe cómo se conectan las tasks y las funciones del workflow. También es una función de Python, decorada con `@flow`, que puede recibir parámetros y devolver resultados. Cada invocación crea un **flow run**.

```python
from prefect import flow

@flow(name="resumen-nyc-taxi")
def resumir(datos):
    filas = contar_filas(datos)
    return {"filas": filas}
```

Un flow aporta nombre, parámetros validados, estados, duración y relación entre tasks. Puede contener decisiones normales de Python como `if`, `for` y manejo de excepciones. Un **subflow** es un flow llamado desde otro flow: lo observaremos en la demostración de perros y gatos. El pipeline principal de NYC Taxi usa un solo flow; workers, work pools, deployments y schedules quedan fuera de esta clase.

## 2.3. Task, flow y función pura

| Elemento | Responsabilidad en NYC Taxi |
|---|---|
| función pura | construir un `Pipeline`, calcular una configuración o validar una regla simple |
| task | descargar datos, preparar un periodo, ejecutar HPO, entrenar y evaluar |
| subflow | agrupar una parte reutilizable que también necesita su propio flow run |
| flow | coordinar meses, familias, decisiones y ranking |

Hacer una sola task que contenga todo el script produce muy poca observabilidad. Convertir cada línea en task fragmenta el código sin aportar significado. Las fronteras deben seguir etapas que una persona pueda nombrar, verificar y diagnosticar.

## 2.4. Estados, reintentos y logs

Durante una ejecución, Prefect registra estados. Para esta clase importan `Pending`, `Running`, `Completed`, `Failed` y `Retrying`. Dentro de una task usamos `get_run_logger()` para asociar mensajes con su ejecución.

Esto permite distinguir tres capas de observabilidad:

- **Prefect:** qué etapa corrió, cuándo y en qué estado terminó;
- **MLflow Tracking:** con qué parámetros se entrenó y qué métrica obtuvo;
- **Model Registry:** qué versiones existen y qué alias tiene cada una.

## 2.5. Instalación y servidor local de Prefect

Prefect se distribuye como paquete de Python. En un proyecto nuevo, su documentación oficial muestra `uv add prefect` para declararlo en `pyproject.toml`. **No ejecutes ese comando en este repositorio:** Prefect ya está declarado como `prefect>=3,<4` y `uv.lock` fija Prefect 3.8.7, la versión estable más reciente verificada el 5 de octubre de 2026. Aquí basta con `uv sync --locked`.

Comprueba que el ambiente sincronizado usa la versión esperada:

```bash
uv run prefect version
```

En una terminal separada inicia el servidor local:

```bash
uv run prefect server start
```

La terminal permanece ocupada mientras el servidor está activo. Déjala abierta y visita [http://127.0.0.1:4200](http://127.0.0.1:4200) en el navegador. Esa es la interfaz local predeterminada; la API vive en `http://127.0.0.1:4200/api`.

En una segunda terminal, desde la raíz del repositorio, indica al cliente qué API debe usar:

```bash
uv run prefect config set PREFECT_API_URL="http://127.0.0.1:4200/api"
```

Después ejecuta el archivo o las celdas con flows. Para detener el servidor vuelve a la primera terminal y presiona `Ctrl+C`. Cliente y servidor usan el mismo ambiente y, por tanto, la misma versión.

**Checkpoint:** `uv run prefect version` muestra 3.8.7, la interfaz abre y una ejecución aparece en **Flow Runs**. Si el servidor no inicia, conserva el código y revisa los logs de la terminal; no cambies a Prefect Cloud durante la clase.

## 2.6. Demostración — datos de perros y gatos

Un **subflow** es un flow llamado desde otro flow. El ejemplo consulta dos APIs independientes: un subflow obtiene un dato de gatos y otro obtiene un dato de perros. El flow padre coordina ambos resultados.

La task `solicitar_json()` concentra la operación externa y permite reintentarla. Los subflows traducen cada respuesta a un texto; el flow padre combina las salidas. Si una API no responde después de los reintentos, el ejemplo usa un texto local para que podamos seguir inspeccionando la estructura.

Ejecuta las dos celdas y busca en la interfaz un flow padre, dos subflows y sus tasks.

In [ ]:
import httpx
from prefect import flow, task
from prefect import get_run_logger as get_demo_logger


@task(name="solicitar-json", retries=2, retry_delay_seconds=1)
def solicitar_json(url: str) -> dict:
    respuesta = httpx.get(url, timeout=10)
    respuesta.raise_for_status()
    return respuesta.json()


@flow(name="Dato de gato")
def obtener_dato_gato() -> str:
    try:
        datos = solicitar_json("https://catfact.ninja/fact?max_length=140")
        return datos["fact"]
    except Exception as error:
        get_demo_logger().warning("API de gatos no disponible: %s", error)
        return "Los gatos duermen gran parte del día."


@flow(name="Dato de perro")
def obtener_dato_perro() -> str:
    try:
        datos = solicitar_json("https://dogapi.dog/api/v2/facts")
        return datos["data"][0]["attributes"]["body"]
    except Exception as error:
        get_demo_logger().warning("API de perros no disponible: %s", error)
        return "Los perros pueden aprender señales verbales y gestuales."


@flow(name="Datos de animales", log_prints=True)
def mostrar_datos_animales() -> dict[str, str]:
    resultados = {
        "gato": obtener_dato_gato(),
        "perro": obtener_dato_perro(),
    }
    print(f"Gato: {resultados['gato']}")
    print(f"Perro: {resultados['perro']}")
    return resultados


In [ ]:
mostrar_datos_animales()


# 3. Orquestar el pipeline actual de NYC Taxi

El pipeline conserva el contrato construido desde la Clase 6 y ampliado en la Tarea 5.

- **Numéricas:** `distancia_km`, `pasajeros`, `hora_recoleccion`.
- **Categóricas:** `zona_origen`, `zona_destino`.
- **Target:** `duracion_minutos`.
- **Familias:** `LinearRegression`, `RandomForestRegressor` y `GradientBoostingRegressor`.

Las zonas permanecen separadas y se codifican con `OneHotEncoder(handle_unknown="ignore")`. El preprocesamiento vive dentro de `sklearn.pipeline.Pipeline`; no se ajusta antes de dividir datos ni se duplica por familia.

Dividiremos el workflow en siete steps observables. Cada subsección explica el objetivo, la entrada y la salida del step correspondiente:

1. asegurar que los Parquet de marzo y abril existan;
2. cargar, preparar y validar ambos meses con el mismo contrato;
3. dividir únicamente marzo para desarrollo y tuning;
4. ejecutar HPO o recuperar parámetros persistidos;
5. entrenar un modelo final por familia con marzo completo;
6. evaluar cada modelo una sola vez con abril;
7. registrar los tres modelos finales, ordenarlos por RMSE y asignar aliases.

El flow principal conecta esos steps y hace visible qué etapa se completó, falló o se reintentó.

## 3.1. Recorrido completo y límites de datos

El recorrido se lee en este orden:

1. `descargar_datos()` garantiza que existan los Parquet de marzo y abril.
2. `cargar_parquet()` y `preparar_y_validar()` producen dos tablas con el mismo contrato.
3. `dividir_desarrollo()` divide solamente marzo en entrenamiento interno y validación de tuning.
4. `obtener_parametros()` ejecuta HPO con esas dos particiones o lee `mejores-parametros.json`.
5. `entrenar_modelo_final()` vuelve a entrenar cada familia con marzo completo.
6. `evaluar_modelo()` consulta abril por primera y única vez.
7. `registrar_modelo_final()` crea tres versiones y `construir_ranking_y_aliases()` asigna sus roles.

La regla crítica es que abril no entra a `dividir_desarrollo()`, `ajustar_familia()` ni `obtener_parametros()`. Sólo aparece en `evaluar_modelo()` después de fijar los parámetros y entrenar el modelo.

### Configuración compartida

El experimento de la clase usa el nombre fijo `/Shared/pcd-otono-2026-clase-13-prefect`. Así no se mezcla con la Clase 12 ni con la Tarea 5. Cada ejecución agrega un identificador a sus runs finales y el modelo registrado termina en `nyc_taxi_trip_duration_clase_13`.

In [ ]:
import json
import sys
from pathlib import Path
from time import perf_counter
from urllib.request import urlretrieve

import mlflow
import mlflow.sklearn
import optuna
import pandas as pd
from dotenv import load_dotenv
from mlflow import MlflowClient
from mlflow.models import infer_signature
from optuna.samplers import TPESampler
from prefect import flow, get_run_logger, task
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

EXPERIMENTO_MLFLOW = "/Shared/pcd-otono-2026-clase-13-prefect"
TRACKING_URI = "databricks"
REGISTRY_URI = "databricks-uc"
CATALOGO = "reemplaza_con_tu_catalogo"
IDENTIFICADOR_RUN = "lince-verde"
HACER_TUNING = True
MAX_FILAS = 20_000
COLUMNAS_FUENTE = [
    "lpep_pickup_datetime", "lpep_dropoff_datetime",
    "trip_distance", "passenger_count",
    "PULocationID", "DOLocationID",
]
FEATURES_NUMERICAS = ["distancia_km", "pasajeros", "hora_recoleccion"]
FEATURES_CATEGORICAS = ["zona_origen", "zona_destino"]
FEATURES = FEATURES_NUMERICAS + FEATURES_CATEGORICAS
TARGET = "duracion_minutos"
FAMILIAS = ("linear_regression", "random_forest", "gradient_boosting")
ALIASES = ("champion", "challenger", "candidate")
URLS = {
    "marzo": "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-03.parquet",
    "abril": "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-04.parquet",
}
NOMBRES = {
    "marzo": "green_tripdata_2026-03.parquet",
    "abril": "green_tripdata_2026-04.parquet",
}


def configurar_mlflow() -> None:
    load_dotenv(override=True)
    mlflow.set_tracking_uri(TRACKING_URI)
    mlflow.set_registry_uri(REGISTRY_URI)
    mlflow.set_experiment(EXPERIMENTO_MLFLOW)


## 3.2. Step 1 — Asegurar los datos

**Objetivo:** entregar rutas locales válidas sin volver a descargar archivos existentes. **Entrada:** el directorio de datos. **Salida:** un diccionario con las rutas de marzo y abril.

`descargar_datos()` recibe un directorio y garantiza que los dos Parquet estén disponibles. No necesitamos otra función `descargar_si_falta`: la misma task decide si reutiliza o descarga cada archivo.

La descarga admite reintentos por ser una operación externa. `cargar_parquet()` queda separado: leer el archivo confirma que su contenido es utilizable.

In [ ]:
@task(name="asegurar-datos", retries=2, retry_delay_seconds=5)
def descargar_datos(directorio: Path) -> dict[str, Path]:
    logger = get_run_logger()
    directorio.mkdir(parents=True, exist_ok=True)
    rutas: dict[str, Path] = {}
    for periodo, url in URLS.items():
        ruta = directorio / NOMBRES[periodo]
        if ruta.exists():
            logger.info("%s ya existe; se reutiliza", ruta.name)
        else:
            logger.info("Descargando %s", ruta.name)
            urlretrieve(url, ruta)
        rutas[periodo] = ruta
    return rutas


## 3.3. Step 2 — Cargar, preparar y validar ambos meses

**Objetivo:** producir dos tablas compatibles con exactamente las mismas cinco features y el mismo target. **Entrada:** un Parquet y la etiqueta del periodo. **Salida:** un `DataFrame` que cumple el contrato.

`COLUMNAS_FUENTE` y `FEATURES` no son lo mismo. Las primeras son los seis campos crudos que leemos del Parquet de TLC; las segundas son las cinco variables con las que entrenamos desde la Clase 6. `preparar_y_validar()` conecta ambos contratos:

- `trip_distance` → `distancia_km`;
- `passenger_count` → `pasajeros`;
- hora de `lpep_pickup_datetime` → `hora_recoleccion`;
- `PULocationID` y `DOLocationID` → `zona_origen` y `zona_destino`;
- diferencia entre dropoff y pickup → `duracion_minutos`, el target.

Después aplica los mismos rangos de la Clase 6, elimina nulos y devuelve exclusivamente `FEATURES + [TARGET]`. Ningún modelo recibe directamente las columnas TLC.

La task recibe `periodo` sólo para producir logs y errores claros. No transforma marzo y abril de manera diferente. Un schema inválido falla sin reintentos porque repetir la misma entrada no crea la columna ausente.

In [ ]:
@task(name="cargar-parquet", task_run_name="cargar-{periodo}")
def cargar_parquet(ruta: Path, periodo: str) -> pd.DataFrame:
    return pd.read_parquet(ruta, columns=COLUMNAS_FUENTE)


@task(name="preparar-y-validar", task_run_name="preparar-{periodo}")
def preparar_y_validar(
    viajes: pd.DataFrame, periodo: str, max_filas: int
) -> pd.DataFrame:
    logger = get_run_logger()
    faltantes = sorted(set(COLUMNAS_FUENTE) - set(viajes.columns))
    if faltantes:
        raise ValueError(f"{periodo}: faltan columnas {faltantes}")

    preparados = viajes.copy()
    preparados[TARGET] = (
        preparados["lpep_dropoff_datetime"]
        - preparados["lpep_pickup_datetime"]
    ).dt.total_seconds() / 60
    preparados["distancia_km"] = preparados["trip_distance"] * 1.60934
    preparados["pasajeros"] = preparados["passenger_count"]
    preparados["hora_recoleccion"] = preparados["lpep_pickup_datetime"].dt.hour
    preparados["zona_origen"] = preparados["PULocationID"]
    preparados["zona_destino"] = preparados["DOLocationID"]

    validos = (
        preparados[TARGET].between(1, 60)
        & preparados["distancia_km"].between(0.1, 100)
        & preparados["pasajeros"].between(1, 6)
        & preparados["zona_origen"].gt(0)
        & preparados["zona_destino"].gt(0)
    )
    preparados = preparados.loc[validos, FEATURES + [TARGET]].dropna().copy()
    if preparados.empty:
        raise ValueError(f"{periodo}: no quedaron filas válidas")
    enteras = ["pasajeros", "hora_recoleccion", "zona_origen", "zona_destino"]
    preparados[enteras] = preparados[enteras].astype(int)
    if len(preparados) > max_filas:
        preparados = preparados.sample(max_filas, random_state=42)
    logger.info("%s quedó con %s filas válidas", periodo, len(preparados))
    return preparados.reset_index(drop=True)


## 3.4. Step 3 — Dividir marzo

**Objetivo:** crear una partición de entrenamiento interno y otra de validación para comparar hiperparámetros sin consultar abril. **Entrada:** marzo preparado. **Salida:** `marzo_train` y `marzo_tuning`, con `random_state=42` para repetir la división.

Abril no aparece en la firma de esta task. Después del HPO, el modelo final se vuelve a ajustar con marzo completo; la partición interna sólo sirve para elegir configuración.

In [ ]:
@task(name="dividir-marzo")
def dividir_desarrollo(
    marzo: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    return train_test_split(marzo, test_size=0.2, random_state=42)


def construir_pipeline(familia: str, parametros: dict) -> Pipeline:
    preprocesamiento = ColumnTransformer(
        [
            ("numericas", "passthrough", FEATURES_NUMERICAS),
            (
                "categoricas",
                OneHotEncoder(handle_unknown="ignore"),
                FEATURES_CATEGORICAS,
            ),
        ]
    )
    estimadores = {
        "linear_regression": LinearRegression,
        "random_forest": RandomForestRegressor,
        "gradient_boosting": GradientBoostingRegressor,
    }
    if familia not in estimadores:
        raise ValueError(f"Familia no reconocida: {familia}")
    estimador = estimadores[familia](**parametros)
    return Pipeline(
        [("preprocesamiento", preprocesamiento), ("modelo", estimador)]
    )


## 3.5. Step 4 — Obtener parámetros con HPO activable

**Objetivo:** devolver una configuración para cada familia sin alterar el resto del workflow. **Entradas:** familia, flag, particiones de marzo y archivo de parámetros. **Salida:** un diccionario de hiperparámetros.

| Valor | Comportamiento |
|---|---|
| `True` | ejecuta los tres estudios con las dos particiones de marzo y guarda mejores parámetros |
| `False` | reutiliza `mejores-parametros.json` y evita repetir trials durante una depuración |

En ambos caminos se construye un pipeline nuevo, se entrena con marzo completo y se evalúa con abril. Las tres familias usan `TPESampler` con la misma semilla y seis trials; lo que cambia es el espacio propuesto para cada modelo. Los trials registran parámetros y RMSE; los modelos completos se registran sólo en los runs finales.

In [ ]:
def proponer_parametros(trial: optuna.Trial, familia: str) -> dict:
    if familia == "linear_regression":
        return {
            "fit_intercept": trial.suggest_categorical(
                "fit_intercept", [True, False]
            )
        }
    if familia == "random_forest":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 50, 150, step=50),
            "max_depth": trial.suggest_int("max_depth", 6, 18, step=6),
            "min_samples_leaf": trial.suggest_int(
                "min_samples_leaf", 2, 8, step=2
            ),
            "random_state": 42,
            "n_jobs": -1,
        }
    if familia == "gradient_boosting":
        return {
            "n_estimators": trial.suggest_int("n_estimators", 50, 150, step=50),
            "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.15),
            "max_depth": trial.suggest_int("max_depth", 2, 4),
            "min_samples_leaf": trial.suggest_int(
                "min_samples_leaf", 2, 8, step=2
            ),
            "random_state": 42,
        }
    raise ValueError(f"Familia no reconocida: {familia}")


def ajustar_familia(
    familia: str,
    marzo_train: pd.DataFrame,
    marzo_tuning: pd.DataFrame,
) -> dict:
    n_trials = 6
    estudio = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=42, n_startup_trials=3),
    )

    def objective(trial: optuna.Trial) -> float:
        parametros = proponer_parametros(trial, familia)
        pipeline = construir_pipeline(familia, parametros)
        inicio = perf_counter()
        pipeline.fit(marzo_train[FEATURES], marzo_train[TARGET])
        tiempo = perf_counter() - inicio
        predicciones = pipeline.predict(marzo_tuning[FEATURES])
        rmse = root_mean_squared_error(marzo_tuning[TARGET], predicciones)
        with mlflow.start_run(
            run_name=f"trial-{trial.number:03d}", nested=True
        ):
            mlflow.log_params(parametros)
            mlflow.log_metrics(
                {"tuning_rmse": rmse, "training_time_seconds": tiempo}
            )
            mlflow.set_tags(
                {
                    "class": "13",
                    "role": "tuning-trial",
                    "model_family": familia,
                }
            )
        return rmse

    with mlflow.start_run(run_name=f"clase-13-tuning-{familia}"):
        mlflow.log_params(
            {
                "optimizer": "optuna-tpe",
                "n_trials": n_trials,
                "objective_metric": "tuning_rmse",
            }
        )
        mlflow.set_tags(
            {"class": "13", "role": "tuning-study", "model_family": familia}
        )
        estudio.optimize(objective, n_trials=n_trials)
        mejores = dict(estudio.best_trial.params)
        mlflow.log_metric("best_tuning_rmse", estudio.best_value)
        mlflow.log_param("best_trial_number", estudio.best_trial.number)
        mlflow.log_params(
            {f"best_{clave}": valor for clave, valor in mejores.items()}
        )

    if familia in {"random_forest", "gradient_boosting"}:
        mejores["random_state"] = 42
    if familia == "random_forest":
        mejores["n_jobs"] = -1
    return mejores


def leer_parametros(ruta: Path) -> dict[str, dict]:
    if not ruta.exists():
        return {}
    return json.loads(ruta.read_text(encoding="utf-8"))


def guardar_parametros(ruta: Path, parametros: dict[str, dict]) -> None:
    ruta.parent.mkdir(parents=True, exist_ok=True)
    ruta.write_text(
        json.dumps(parametros, indent=2, sort_keys=True), encoding="utf-8"
    )


@task(name="obtener-parametros", task_run_name="parametros-{familia}")
def obtener_parametros(
    familia: str,
    hacer_tuning: bool,
    marzo_train: pd.DataFrame,
    marzo_tuning: pd.DataFrame,
    ruta_parametros: Path,
) -> dict:
    logger = get_run_logger()
    parametros = leer_parametros(ruta_parametros)
    if hacer_tuning:
        logger.info("HPO activado para %s", familia)
        parametros[familia] = ajustar_familia(
            familia, marzo_train, marzo_tuning
        )
        guardar_parametros(ruta_parametros, parametros)
        return parametros[familia]
    if familia not in parametros:
        raise FileNotFoundError(
            f"Faltan parámetros para {familia}; usa HACER_TUNING = True"
        )
    logger.info("Reutilizando parámetros de %s", ruta_parametros.name)
    return parametros[familia]


## 3.6. Step 5 — Entrenar los modelos finales

**Objetivo:** ajustar un `sklearn.pipeline.Pipeline` nuevo por familia usando marzo completo y los parámetros elegidos. **Entradas:** familia, parámetros y marzo. **Salida:** un pipeline ajustado.

Usar marzo completo recupera las filas reservadas durante HPO. El preprocesamiento permanece dentro del objeto de scikit-learn, por lo que `fit()` aprende la codificación y el estimador como una sola unidad.

In [ ]:
@task(name="entrenar-modelo", task_run_name="entrenar-{familia}")
def entrenar_modelo_final(
    familia: str, parametros: dict, marzo: pd.DataFrame
) -> dict:
    pipeline = construir_pipeline(familia, parametros)
    inicio = perf_counter()
    pipeline.fit(marzo[FEATURES], marzo[TARGET])
    return {
        "familia": familia,
        "parametros": parametros,
        "pipeline": pipeline,
        "training_time_seconds": perf_counter() - inicio,
    }


## 3.7. Step 6 — Evaluar una vez con abril

**Objetivo:** estimar el desempeño final de cada candidato sin retroalimentar HPO. **Entradas:** modelo ya entrenado y abril preparado. **Salida:** RMSE, tiempo de entrenamiento y metadatos de la familia.

Abril se usa después de fijar hiperparámetros. Si su RMSE se usa para volver a ajustar la búsqueda, deja de funcionar como validación final.

In [ ]:
@task(name="evaluar-modelo", task_run_name="evaluar-{familia}")
def evaluar_modelo(
    familia: str, entrenamiento: dict, abril: pd.DataFrame
) -> dict:
    predicciones = entrenamiento["pipeline"].predict(abril[FEATURES])
    return {
        **entrenamiento,
        "validation_rmse": root_mean_squared_error(
            abril[TARGET], predicciones
        ),
    }


## 3.8. Step 7 — Registrar, ordenar y asignar aliases

**Objetivo:** conservar únicamente los tres candidatos finales y convertir sus métricas en roles operativos. **Entradas:** modelo, parámetros y RMSE de cada familia. **Salida:** versiones registradas y ranking con aliases.

Cada familia devuelve nombre, parámetros, RMSE final, tiempo, `run_id` y `model_uri`. Prefect coordina estado operativo; MLflow conserva el experimento. El ranking ascendente asigna:

```text
menor RMSE → champion
segundo    → challenger
tercero    → candidate
```

Los aliases son roles dinámicos, no nombres alternativos de algoritmos. Registry recibe únicamente los tres modelos finales, no todos los trials.

In [ ]:
@task(name="registrar-modelo", task_run_name="registrar-{familia}")
def registrar_modelo_final(
    familia: str,
    evaluacion: dict,
    marzo: pd.DataFrame,
    abril: pd.DataFrame,
    modelo_registrado: str,
    identificador_run: str,
) -> dict:
    pipeline = evaluacion["pipeline"]
    ejemplo = abril[FEATURES].head(5)
    firma = infer_signature(ejemplo, pipeline.predict(ejemplo))
    dataset_marzo = mlflow.data.from_pandas(
        marzo, source=NOMBRES["marzo"], targets=TARGET,
        name="green-taxi-2026-03-completo",
    )
    dataset_abril = mlflow.data.from_pandas(
        abril, source=NOMBRES["abril"], targets=TARGET,
        name="green-taxi-2026-04-final",
    )
    with mlflow.start_run(
        run_name=f"clase-13-{identificador_run}-{familia}-final"
    ) as run:
        mlflow.log_input(dataset_marzo, context="training")
        mlflow.log_input(dataset_abril, context="final-validation")
        mlflow.log_params(evaluacion["parametros"])
        mlflow.log_metrics(
            {
                "validation_rmse": evaluacion["validation_rmse"],
                "training_time_seconds": evaluacion["training_time_seconds"],
            }
        )
        mlflow.set_tags(
            {
                "class": "13", "role": "family-final",
                "model_family": familia, "prefect_run": identificador_run,
            }
        )
        modelo = mlflow.sklearn.log_model(
            pipeline, name="model", input_example=ejemplo, signature=firma,
            serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_PICKLE,
        )
    version = mlflow.register_model(
        model_uri=f"models:/{modelo.model_id}", name=modelo_registrado
    )
    return {
        "model_family": familia,
        "validation_rmse": evaluacion["validation_rmse"],
        "training_time_seconds": evaluacion["training_time_seconds"],
        "run_id": run.info.run_id,
        "model_id": modelo.model_id,
        "version": str(version.version),
    }


@task(name="asignar-aliases")
def construir_ranking_y_aliases(
    resultados: list[dict], modelo_registrado: str
) -> pd.DataFrame:
    ranking = (
        pd.DataFrame(resultados)
        .sort_values("validation_rmse")
        .reset_index(drop=True)
    )
    ranking["alias"] = list(ALIASES)
    cliente = MlflowClient(
        tracking_uri=TRACKING_URI, registry_uri=REGISTRY_URI
    )
    for fila in ranking.itertuples(index=False):
        cliente.set_registered_model_alias(
            name=modelo_registrado, alias=fila.alias, version=fila.version
        )
        cliente.set_model_version_tag(
            name=modelo_registrado, version=fila.version,
            key="validation_rmse", value=f"{fila.validation_rmse:.6f}",
        )
    return ranking


## 3.9. Flow principal y ejecución

El flow ejecuta las familias secuencialmente para que los contextos de MLflow sean fáciles de inspeccionar. Para mantener el script simple, las cuatro decisiones de ejecución están declaradas como constantes al inicio: catálogo, identificador anónimo, activación de HPO y máximo de filas.

In [ ]:
@flow(
    name="nyc-taxi-clase-13-prefect",
    flow_run_name="clase-13-{identificador_run}",
)
def flujo_entrenamiento(
    modelo_registrado: str,
    identificador_run: str,
    directorio_datos: Path,
    ruta_parametros: Path,
    hacer_tuning: bool = False,
    max_filas: int = 20_000,
) -> pd.DataFrame:
    configurar_mlflow()
    rutas = descargar_datos(directorio_datos)
    marzo = preparar_y_validar(
        cargar_parquet(rutas["marzo"], "marzo"), "marzo", max_filas
    )
    abril = preparar_y_validar(
        cargar_parquet(rutas["abril"], "abril"), "abril", max_filas
    )
    marzo_train, marzo_tuning = dividir_desarrollo(marzo)

    resultados = []
    for familia in FAMILIAS:
        parametros = obtener_parametros(
            familia, hacer_tuning, marzo_train, marzo_tuning, ruta_parametros
        )
        entrenamiento = entrenar_modelo_final(familia, parametros, marzo)
        evaluacion = evaluar_modelo(familia, entrenamiento, abril)
        resultados.append(
            registrar_modelo_final(
                familia, evaluacion, marzo, abril,
                modelo_registrado, identificador_run,
            )
        )
    return construir_ranking_y_aliases(resultados, modelo_registrado)


if __name__ == "__main__" and "ipykernel" not in sys.modules:
    raiz = Path.cwd()
    trabajo = raiz / "labs" / "trabajo-local" / "clase-13"
    nombre_modelo = f"{CATALOGO}.default.nyc_taxi_trip_duration_clase_13"
    ranking_final = flujo_entrenamiento(
        modelo_registrado=nombre_modelo,
        identificador_run=IDENTIFICADOR_RUN,
        directorio_datos=trabajo / "data",
        ruta_parametros=trabajo / "mejores-parametros.json",
        hacer_tuning=HACER_TUNING,
        max_filas=MAX_FILAS,
    )
    print(ranking_final.to_string(index=False))


# 4. Práctica guiada

El resultado de la práctica es un único archivo llamado `pipeline_entrenamiento_prefect.py`. Todo se realiza desde la **raíz del repositorio público de la clase**; no uses `pcd-entregas-2026`, no crees rama y no edites este notebook.

## 4.1. Crear el script

En una terminal ubicada en la raíz del repositorio ejecuta:

```bash
mkdir -p labs/trabajo-local/clase-13
pwd
```

`pwd` debe mostrar la raíz del repositorio, no `labs/trabajo-local/clase-13`. Desde VS Code crea:

```text
labs/trabajo-local/clase-13/pipeline_entrenamiento_prefect.py
```

Copia en ese archivo, en el mismo orden, las celdas de código de las secciones **Configuración compartida** y **3.2–3.9**. No copies las demostraciones de animales. El código mostrado está completo; los únicos cambios solicitados son:

1. en `preparar_y_validar()`, agrega antes de los filtros un log con el periodo y la cantidad de filas recibidas;
2. en `construir_ranking_y_aliases()`, agrega una columna `posicion` con los valores `1`, `2` y `3`.

Verifica que `.env` permanezca ignorado y que contenga las variables de Databricks creadas en la Clase 12:

```bash
git check-ignore .env
```

## 4.2. Terminal 1 — iniciar Prefect

Desde la raíz del repositorio inicia el servidor y deja esta terminal abierta:

```bash
uv run prefect server start
```

Abre [http://127.0.0.1:4200](http://127.0.0.1:4200).

## 4.3. Terminal 2 — configurar y ejecutar

Abre otra terminal en la raíz del repositorio y configura la API local:

```bash
uv run prefect config set PREFECT_API_URL="http://127.0.0.1:4200/api"
```

Consulta Catalog Explorer y, al inicio del script, sustituye `reemplaza_con_tu_catalogo` por el nombre exacto del catálogo, no por el identificador de la URL. Conserva un identificador anónimo reconocible, por ejemplo `lince-verde`, y deja `HACER_TUNING = True` para la primera ejecución:

```bash
uv run python labs/trabajo-local/clase-13/pipeline_entrenamiento_prefect.py
```

Esta corrida crea `mejores-parametros.json`. Después cambia únicamente `IDENTIFICADOR_RUN = "lince-verde-sin-hpo"` y `HACER_TUNING = False`; guarda el archivo y ejecuta exactamente el mismo comando:

```bash
uv run python labs/trabajo-local/clase-13/pipeline_entrenamiento_prefect.py
```

## 4.4. Corroborar las dos ejecuciones

**En Prefect**, abre **Flow Runs** y confirma:

- los flows `clase-13-lince-verde` y `clase-13-lince-verde-sin-hpo` —o los dos identificadores anónimos que elegiste—;
- tasks separadas para datos, preparación, parámetros, entrenamiento, evaluación, registro y aliases;
- todos los estados en `Completed`;
- logs de reutilización de los Parquet y, en la segunda corrida, del JSON de parámetros.

**En Databricks MLflow**, abre el experimento `/Shared/pcd-otono-2026-clase-13-prefect` y confirma:

- en la primera corrida, tres runs `clase-13-tuning-*` con sus `trial-NNN`;
- tres runs finales con el identificador de Prefect, parámetros, RMSE y tiempo;
- el modelo `<catalogo>.default.nyc_taxi_trip_duration_clase_13` con tres versiones nuevas;
- aliases `champion`, `challenger` y `candidate` de acuerdo con el ranking;
- en la segunda corrida, tres nuevos runs finales pero ningún estudio nuevo de HPO.

**Checkpoint final:** la tabla impresa en la terminal incluye `posicion`, familia, RMSE, versión y alias, y la misma corrida puede relacionarse por nombre entre Prefect y MLflow.

# 5. Errores frecuentes

| Síntoma | Causa probable | Siguiente verificación |
|---|---|---|
| flow ausente de la interfaz | API local no configurada | revisa `prefect server start` |
| siempre descarga | ruta relativa desde otro directorio | registra la ruta resuelta |
| abril entra a objective | fuga de información | elimínalo de la firma de HPO |
| categorías inconsistentes | preprocesamiento separado | conserva `OneHotEncoder` en `Pipeline` |
| cada trial guarda modelo | logging excesivo | registra sólo runs finales |
| todos los errores reintentan | retries por costumbre | limita retries a fallas transitorias |
| alias fijo por familia | ranking codificado | ordena por RMSE |
| depende del notebook | variables ocultas | declara imports, configuración y rutas |

# Resumen

- Cron programa un comando; un orquestador representa etapas, dependencias y estados.
- Logging operativo, MLflow Tracking y Registry resuelven problemas distintos.
- `sklearn.Pipeline` conserva preprocesamiento y modelo; Prefect coordina el workflow.
- Marzo y abril comparten preparación, pero sólo marzo participa en HPO.
- `hacer_tuning=False` reutiliza parámetros; no evita reentrenar ni evaluar.
- La descarga reutiliza archivos existentes.
- Los aliases se asignan por resultados.

# ✅ Check final de la clase

Comprueba que puedes distinguir función, task, subflow, flow y `sklearn.Pipeline`; ejecutar el script desde la raíz del repositorio; justificar reintentos; mostrar que la segunda ejecución reutiliza datos y parámetros; explicar por qué abril no entra al HPO; y relacionar por nombre las mismas dos corridas en Prefect y MLflow.

Este check no constituye una entrega en Canvas.

# Tarea 6 — Orquestar el pipeline de entrenamiento con Prefect

Refactoriza tu propia Tarea 5 como un flow de Prefect. Conserva las tres familias, cinco features, HPO, separación marzo/abril, MLflow y aliases dinámicos.

Consulta requisitos, evidencias y rúbrica en [`docs/tareas/tarea-06-pipeline-prefect.md`](../docs/tareas/tarea-06-pipeline-prefect.md). **Fecha límite: viernes 9 de octubre de 2026 a las 19:55, hora de la Ciudad de México.** La entrega oficial se realiza en Canvas.

## Quiz de la siguiente clase

Al inicio de la Clase 14 habrá un quiz individual sobre los contenidos que todavía no habían sido evaluados por el Quiz 3:

- **Clase 11:** Tracking Server frente a Model Registry, versiones, registro, aliases y carga de modelos;
- **Clase 12:** HPO con Optuna, parent/child runs, separación marzo/abril y recorrido local frente a Databricks;
- **Clase 13:** cron frente a orquestación, logging, función/task/flow, estados, reintentos y diseño del pipeline con Prefect.

# Referencias

- [Prefect: instalación](https://docs.prefect.io/v3/get-started/install)
- [Prefect: servidor local](https://docs.prefect.io/v3/how-to-guides/self-hosted/server-cli)
- [Prefect: escribir y ejecutar workflows](https://docs.prefect.io/latest/tutorial/flows)
- [Prefect 3.8.7 en PyPI](https://pypi.org/project/prefect/3.8.7/)
- [Cat Facts API](https://catfact.ninja/)
- [Dog API](https://dogapi.dog/)
- [Python: logging](https://docs.python.org/3/library/logging.html)
- [scikit-learn: Pipeline](https://scikit-learn.org/stable/modules/compose.html#pipeline)
- [Optuna](https://optuna.readthedocs.io/en/stable/)
- [MLflow: hyperparameter tuning](https://mlflow.org/docs/latest/ml/getting-started/hyperparameter-tuning/)
- [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)